# 07 - Build ClinVar tabular full dataset

Notebook nay tao file Parquet rieng cho tabular/XGBoost, khong dung chung `clinvar_training_metadata.parquet` cua CNN.

Input:

- `Data/variant_summary.txt`

Output:

- `processed/clinvar_tabular_full.parquet`

Dataset nay giu nhieu cot ClinVar goc hon va them cac cot suy ra de train model tabular.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
RAW_VARIANT_SUMMARY = PROJECT_DIR / "Data" / "variant_summary.txt"
PROCESSED_DIR = PROJECT_DIR / "processed"
OUTPUT_PARQUET = PROCESSED_DIR / "clinvar_tabular_full.parquet"

if not RAW_VARIANT_SUMMARY.exists():
    raise FileNotFoundError(RAW_VARIANT_SUMMARY)

pd.set_option("display.max_columns", 120)
RAW_VARIANT_SUMMARY, OUTPUT_PARQUET

## 1. Cau hinh cot va label mapping

Cot `ClinicalSignificance` chi dung de tao `Y`, khong nen dua truc tiep vao feature X.

In [ ]:
USE_COLUMNS = [
    "#AlleleID",
    "Type",
    "Name",
    "GeneID",
    "GeneSymbol",
    "HGNC_ID",
    "ClinicalSignificance",
    "ClinSigSimple",
    "LastEvaluated",
    "RS# (dbSNP)",
    "RCVaccession",
    "PhenotypeIDS",
    "PhenotypeList",
    "Origin",
    "OriginSimple",
    "Assembly",
    "ChromosomeAccession",
    "Chromosome",
    "Start",
    "Stop",
    "ReferenceAllele",
    "AlternateAllele",
    "Cytogenetic",
    "ReviewStatus",
    "NumberSubmitters",
    "Guidelines",
    "TestedInGTR",
    "OtherIDs",
    "SubmitterCategories",
    "VariationID",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "SCVsForAggregateGermlineClassification",
]

KEY_COLUMNS = ["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"]
REQUIRED_COLUMNS = [
    "ClinicalSignificance",
    "Assembly",
    "Chromosome",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "Type",
    "ReviewStatus",
    "GeneSymbol",
]

positive_labels = {"Pathogenic", "Likely pathogenic"}
negative_labels = {"Benign", "Likely benign"}


def split_clinical_significance(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("/") if part.strip()]


def map_clean_label(value):
    labels = split_clinical_significance(value)
    label_set = set(labels)
    if not labels:
        return pd.NA
    if label_set <= positive_labels:
        return 1
    if label_set <= negative_labels:
        return 0
    return pd.NA


def simplify_review_status(value):
    value = str(value).lower()
    if "practice guideline" in value:
        return "practice_guideline"
    if "reviewed by expert panel" in value:
        return "expert_panel"
    if "multiple submitters" in value and "no conflicts" in value:
        return "multiple_no_conflicts"
    if "single submitter" in value:
        return "single_submitter"
    if "conflicting" in value:
        return "conflicting"
    if "criteria provided" in value:
        return "criteria_provided_other"
    return "other"

## 2. Helper tao feature suy ra

In [ ]:
transition_pairs = {"A>G", "G>A", "C>T", "T>C"}


def parse_year(series):
    parsed = pd.to_datetime(series.replace({"-": pd.NA, "na": pd.NA}), errors="coerce")
    return parsed.dt.year.astype("Int16")


def add_derived_features(df):
    out = df.copy()

    out["Y"] = out["ClinicalSignificance"].map(map_clean_label)
    out = out.dropna(subset=["Y"]).copy()
    out["Y"] = out["Y"].astype("int8")

    out["ReviewStatusSimple"] = out["ReviewStatus"].map(simplify_review_status)
    out["ref_alt"] = out["ReferenceAlleleVCF"] + ">" + out["AlternateAlleleVCF"]
    out["is_transition"] = out["ref_alt"].isin(transition_pairs).astype("int8")
    out["is_transversion"] = (1 - out["is_transition"]).astype("int8")

    out["PositionVCF_int"] = pd.to_numeric(out["PositionVCF"], errors="coerce").astype("Int64")
    out["Start_int"] = pd.to_numeric(out["Start"], errors="coerce").astype("Int64")
    out["Stop_int"] = pd.to_numeric(out["Stop"], errors="coerce").astype("Int64")
    out["variant_span"] = (out["Stop_int"] - out["Start_int"] + 1).astype("Int64")

    out["NumberSubmitters_int"] = pd.to_numeric(out["NumberSubmitters"], errors="coerce").astype("Int16")
    out["SubmitterCategories_int"] = pd.to_numeric(out["SubmitterCategories"], errors="coerce").astype("Int16")
    out["LastEvaluated_year"] = parse_year(out["LastEvaluated"])

    out["has_rs"] = out["RS# (dbSNP)"].notna() & ~out["RS# (dbSNP)"].isin(["-", "na", ""])
    out["has_rs"] = out["has_rs"].astype("int8")
    out["has_omim_other_id"] = out["OtherIDs"].fillna("").str.contains("OMIM", case=False, regex=False).astype("int8")
    out["tested_in_gtr_flag"] = out["TestedInGTR"].eq("Y").astype("int8")

    out["phenotype_count"] = out["PhenotypeList"].fillna("").map(lambda x: 0 if x in {"", "-"} else len(str(x).split("|"))).astype("int16")
    out["rcv_count"] = out["RCVaccession"].fillna("").map(lambda x: 0 if x in {"", "-"} else len(str(x).split("|"))).astype("int16")
    out["scv_count"] = out["SCVsForAggregateGermlineClassification"].fillna("").map(lambda x: 0 if x in {"", "-"} else len(str(x).split("|"))).astype("int16")

    out["chromosome_is_autosome"] = out["Chromosome"].astype(str).str.fullmatch(r"\d+").fillna(False).astype("int8")
    out["chromosome_is_sex"] = out["Chromosome"].isin(["X", "Y"]).astype("int8")
    out["chromosome_is_mt"] = out["Chromosome"].isin(["MT", "M"]).astype("int8")

    return out

## 3. Doc raw ClinVar theo chunk va filter tabular dataset

Filter giong pipeline SNV sach:

- nhan Y ro rang
- `Assembly == GRCh38`
- `Type == single nucleotide variant`
- `ReviewStatus` dang tin cay
- khong thieu key columns va allele A/C/G/T

Khong kiem tra FASTA/ref allele o notebook nay de giu tabular dataset doc lap va nhanh.

In [ ]:
CHUNK_SIZE = 500_000
trusted_review_patterns = ["criteria provided", "reviewed by expert panel", "practice guideline"]
chunks = []
raw_rows = 0
kept_rows = 0

reader = pd.read_csv(
    RAW_VARIANT_SUMMARY,
    sep="	",
    dtype=str,
    usecols=USE_COLUMNS,
    chunksize=CHUNK_SIZE,
    compression="infer",
)

for chunk in tqdm(reader, desc="read/filter ClinVar", unit="chunk"):
    raw_rows += len(chunk)

    chunk = chunk.dropna(subset=REQUIRED_COLUMNS).copy()
    missing_token_mask = chunk[REQUIRED_COLUMNS].isin(["", "-", "na", "NA"]).any(axis=1)
    chunk = chunk.loc[~missing_token_mask].copy()

    chunk = chunk.loc[chunk["Assembly"].eq("GRCh38")].copy()
    chunk = chunk.loc[chunk["Type"].eq("single nucleotide variant")].copy()

    review_status = chunk["ReviewStatus"].fillna("")
    trusted_review_mask = review_status.str.contains("|".join(trusted_review_patterns), case=False, regex=True)
    low_confidence_review_mask = review_status.str.fullmatch("no assertion criteria provided", case=False)
    chunk = chunk.loc[trusted_review_mask & ~low_confidence_review_mask].copy()

    snv_base_mask = (
        chunk["ReferenceAlleleVCF"].str.fullmatch("[ACGT]")
        & chunk["AlternateAlleleVCF"].str.fullmatch("[ACGT]")
        & chunk["PositionVCF"].str.fullmatch(r"\d+")
    )
    chunk = chunk.loc[snv_base_mask].copy()

    chunk = add_derived_features(chunk)
    if len(chunk):
        chunks.append(chunk)
        kept_rows += len(chunk)

print(f"raw_rows={raw_rows:,}")
print(f"kept_rows={kept_rows:,}")
print(f"chunks={len(chunks):,}")

In [ ]:
clinvar_tabular_df = pd.concat(chunks, ignore_index=True)

# Drop exact duplicate genomic alleles if present; keep first rich ClinVar row.
clinvar_tabular_df = clinvar_tabular_df.drop_duplicates(subset=KEY_COLUMNS, keep="first").copy()

print(clinvar_tabular_df.shape)
display(clinvar_tabular_df["Y"].value_counts())
display(clinvar_tabular_df.head())

## 4. Luu Parquet rieng cho tabular/XGBoost

## 4B. Optional: tao ban mot dong moi variant

File `clinvar_tabular_full.parquet` giu nhieu record ClinVar, co the co nhieu dong cho cung genomic allele do phenotype/RCV khac nhau.

Neu can model tabular mot dong/mot allele, tao them:

`processed/clinvar_tabular_unique_variant.parquet`

In [ ]:
UNIQUE_OUTPUT_PARQUET = PROCESSED_DIR / "clinvar_tabular_unique_variant.parquet"

clinvar_tabular_unique_df = (
    clinvar_tabular_df
    .sort_values(["NumberSubmitters_int", "ReviewStatusSimple"], ascending=[False, True])
    .drop_duplicates(subset=KEY_COLUMNS, keep="first")
    .copy()
)

clinvar_tabular_unique_df.to_parquet(UNIQUE_OUTPUT_PARQUET, index=False, engine="pyarrow")

print(f"Saved: {UNIQUE_OUTPUT_PARQUET}")
print(f"Full rows: {len(clinvar_tabular_df):,}")
print(f"Unique variant rows: {len(clinvar_tabular_unique_df):,}")
display(clinvar_tabular_unique_df["Y"].value_counts())

In [ ]:
clinvar_tabular_df.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow")

check_df = pd.read_parquet(OUTPUT_PARQUET)
print(f"Saved: {OUTPUT_PARQUET}")
print(f"Shape: {check_df.shape}")
print(f"Size MB: {OUTPUT_PARQUET.stat().st_size / 1024 / 1024:.2f}")
display(check_df.head())

## 5. Cot goi y cho XGBoost

Khong dua `ClinicalSignificance`, `ClinSigSimple` vao feature X vi day la label/leakage.

In [ ]:
do_not_use_as_features = ["ClinicalSignificance", "ClinSigSimple", "Y"]
recommended_low_leakage_features = [
    "Chromosome",
    "PositionVCF_int",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "ref_alt",
    "Type",
    "OriginSimple",
    "ReviewStatusSimple",
    "NumberSubmitters_int",
    "SubmitterCategories_int",
    "is_transition",
    "is_transversion",
    "variant_span",
    "has_rs",
    "tested_in_gtr_flag",
    "LastEvaluated_year",
]
recommended_pragmatic_features = recommended_low_leakage_features + [
    "GeneSymbol",
    "GeneID",
    "HGNC_ID",
    "Cytogenetic",
    "phenotype_count",
    "rcv_count",
    "scv_count",
    "has_omim_other_id",
]

print("do_not_use_as_features:")
print(do_not_use_as_features)
print("\nrecommended_low_leakage_features:")
print(recommended_low_leakage_features)
print("\nrecommended_pragmatic_features:")
print(recommended_pragmatic_features)